In [7]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import time

# Pustaka untuk Baseline IBCF & Evaluasi
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

# Pustaka untuk Model Usulan (SVD)
from implicit.als import AlternatingLeastSquares

# Pustaka untuk Visualisasi Hasil (Fase 6)
import matplotlib.pyplot as plt
import seaborn as sns

# Muat data Anda (train.csv)
train = pd.read_csv('train.csv')
data_target = pd.read_csv('data_target_users_test.csv')

print("Pustaka dan data berhasil dimuat.")

Pustaka dan data berhasil dimuat.


# Recommendation System with ALS Models

This notebook implements two recommendation models using Alternating Least Squares (ALS):
1. **ALS - Default Parameters** - Without tuning (baseline)
2. **ALS - Tuned Parameters** - With hyperparameter optimization

The goal is to generate top 10 item recommendations for each user and evaluate using MAP@10.
We compare the performance improvement achieved through hyperparameter tuning.

## Step 1: Data Preparation and Split

We'll split the data into training and validation sets to evaluate our models.

In [8]:
# Data preparation
print("Original dataset shape:", train.shape)
print("Number of unique users:", train['user_id'].nunique())
print("Number of unique items:", train['item_id'].nunique())

# Split data: 80% training, 20% validation
train_df, validation_df = train_test_split(train, test_size=0.2, random_state=42)

print("\nAfter split:")
print("Training set size:", len(train_df))
print("Validation set size:", len(validation_df))

# Create validation ground truth (what users actually interacted with)
validation_truth = validation_df.groupby('user_id')['item_id'].apply(list).to_dict()
validation_users = list(validation_truth.keys())

print(f"\nNumber of users in validation: {len(validation_users)}")
print(f"Sample validation truth for user {validation_users[0]}: {validation_truth[validation_users[0]][:3]}...")

Original dataset shape: (269764, 2)
Number of unique users: 13876
Number of unique items: 123069

After split:
Training set size: 215811
Validation set size: 53953

Number of users in validation: 11443
Sample validation truth for user 8: ['1567407781']...


## Step 2: Evaluation Metric - MAP@10

Mean Average Precision at K (MAP@K) is the evaluation metric for ranking quality.

In [9]:
def apk(actual, predicted, k=10):
    """
    Computes the average precision at k.
    
    Parameters:
    -----------
    actual : list
        A list of elements that are to be predicted (ground truth)
    predicted : list
        A list of predicted elements (order matters)
    k : int
        The maximum number of predicted elements
        
    Returns:
    --------
    score : double
        The average precision at k over the input lists
    """
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    if not actual:
        return 0.0

    return score / min(len(actual), k)


def mapk(actual, predicted, k=10):
    """
    Computes the mean average precision at k.
    
    Parameters:
    -----------
    actual : list of lists
        A list of lists of elements that are to be predicted
    predicted : list of lists
        A list of lists of predicted elements
    k : int
        The maximum number of predicted elements
        
    Returns:
    --------
    score : double
        The mean average precision at k over the input lists
    """
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

print("MAP@K evaluation functions defined successfully.")

MAP@K evaluation functions defined successfully.


## Model 2: ALS (Alternating Least Squares) - Default Parameters

ALS is a matrix factorization technique optimized for implicit feedback data.

In [10]:
print("=" * 60)
print("MODEL 2: ALS WITH DEFAULT PARAMETERS")
print("=" * 60)

# Prepare data for ALS
print("\n1. Preparing data for ALS...")

# Create mappings
all_users = sorted(train_df['user_id'].unique())
all_items_als = sorted(train_df['item_id'].unique())

user_id_to_idx = {user_id: idx for idx, user_id in enumerate(all_users)}
item_id_to_idx = {item_id: idx for idx, item_id in enumerate(all_items_als)}

# Create sparse matrix for ALS (item-user matrix for implicit library)
rows = train_df['item_id'].map(item_id_to_idx).values
cols = train_df['user_id'].map(user_id_to_idx).values
data = np.ones(len(train_df))  # Implicit feedback (1 for interaction)

item_user_sparse = csr_matrix((data, (rows, cols)), 
                               shape=(len(all_items_als), len(all_users)))

print(f"   Matrix shape (items x users): {item_user_sparse.shape}")
print(f"   Number of interactions: {item_user_sparse.nnz}")
print(f"   Sparsity: {100 * (1 - item_user_sparse.nnz / (item_user_sparse.shape[0] * item_user_sparse.shape[1])):.4f}%")

MODEL 2: ALS WITH DEFAULT PARAMETERS

1. Preparing data for ALS...
   Matrix shape (items x users): (104852, 13872)
   Number of interactions: 215811
   Sparsity: 99.9852%


In [11]:
print("\n2. Training ALS model with default parameters...")

# Default ALS parameters
ALS_FACTORS_DEFAULT = 50
ALS_REG_DEFAULT = 0.01
ALS_ITER_DEFAULT = 15
ALS_ALPHA_DEFAULT = 1.0

print(f"   Factors: {ALS_FACTORS_DEFAULT}")
print(f"   Regularization: {ALS_REG_DEFAULT}")
print(f"   Iterations: {ALS_ITER_DEFAULT}")
print(f"   Alpha: {ALS_ALPHA_DEFAULT}")

# Create confidence matrix
als_confidence_default = (item_user_sparse * ALS_ALPHA_DEFAULT).astype('float64')

# Initialize and train model
model_als_default = AlternatingLeastSquares(
    factors=ALS_FACTORS_DEFAULT,
    regularization=ALS_REG_DEFAULT,
    iterations=ALS_ITER_DEFAULT,
    random_state=42
)

start_time = time.time()
model_als_default.fit(als_confidence_default)
print(f"   Training time: {time.time() - start_time:.2f} seconds")


2. Training ALS model with default parameters...
   Factors: 50
   Regularization: 0.01
   Iterations: 15
   Alpha: 1.0


c:\Users\perak\AppData\Local\Programs\Python\Python311\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [00:59<00:00,  3.96s/it]

   Training time: 59.55 seconds


In [12]:
print("\n3. Generating recommendations for validation users...")

predicted_list_als_default = []
actual_list_als_default = []

# Store training items for filtering
train_items_by_user_als = train_df.groupby('user_id')['item_id'].apply(set).to_dict()

start_time = time.time()
for i, user in enumerate(validation_users):
    if (i + 1) % 1000 == 0:
        print(f"   Processed {i + 1}/{len(validation_users)} users...")
    
    if user not in user_id_to_idx:
        # Cold start user - cannot generate recommendations
        predicted_list_als_default.append([])
        actual_list_als_default.append(validation_truth[user])
        continue
    
    user_idx = user_id_to_idx[user]
    
    # Get recommendations (returns item indices and scores)
    ids, scores = model_als_default.recommend(
        user_idx, 
        als_confidence_default[user_idx],
        N=10,
        filter_already_liked_items=True
    )
    
    # Convert indices back to item IDs
    recommended_items = [all_items_als[idx] for idx in ids]
    
    predicted_list_als_default.append(recommended_items)
    actual_list_als_default.append(validation_truth[user])

print(f"   Time taken: {time.time() - start_time:.2f} seconds")

# Evaluate ALS Default
print("\n4. Evaluating ALS model with default parameters...")
mapk_score_als_default = mapk(actual_list_als_default, predicted_list_als_default, k=10)

print(f"\n{'='*60}")
print(f"ALS (Default) Results:")
print(f"MAP@10 Score: {mapk_score_als_default:.6f}")
print(f"{'='*60}\n")


3. Generating recommendations for validation users...
   Processed 1000/11443 users...
   Processed 2000/11443 users...
   Processed 3000/11443 users...
   Processed 4000/11443 users...
   Processed 5000/11443 users...
   Processed 6000/11443 users...
   Processed 7000/11443 users...
   Processed 8000/11443 users...
   Processed 9000/11443 users...
   Processed 10000/11443 users...
   Processed 11000/11443 users...
   Time taken: 10.04 seconds

4. Evaluating ALS model with default parameters...

ALS (Default) Results:
MAP@10 Score: 0.000011



## Model 3: ALS with Hyperparameter Tuning

We'll tune the ALS model by testing different combinations of hyperparameters to achieve better MAP@10.

In [13]:
print("=" * 60)
print("MODEL 3: ALS WITH HYPERPARAMETER TUNING")
print("=" * 60)

# Define hyperparameter grid
param_grid = {
    'factors': [50, 100, 150],
    'regularization': [0.01, 0.05, 0.1],
    'iterations': [15, 20, 25],
    'alpha': [1.0, 10.0, 40.0]
}

print("\n1. Hyperparameter Search Grid:")
for param, values in param_grid.items():
    print(f"   {param}: {values}")

# Sample a subset of validation users for faster tuning (use 20%)
np.random.seed(42)
tuning_sample_size = max(500, int(len(validation_users) * 0.2))
tuning_users = np.random.choice(validation_users, size=tuning_sample_size, replace=False)
print(f"\n2. Using {len(tuning_users)} users for hyperparameter tuning")

MODEL 3: ALS WITH HYPERPARAMETER TUNING

1. Hyperparameter Search Grid:
   factors: [50, 100, 150]
   regularization: [0.01, 0.05, 0.1]
   iterations: [15, 20, 25]
   alpha: [1.0, 10.0, 40.0]

2. Using 2288 users for hyperparameter tuning


In [14]:
def evaluate_als_params(factors, regularization, iterations, alpha, 
                        item_user_matrix, user_mapping, item_list, 
                        eval_users, ground_truth):
    """
    Train ALS with given parameters and evaluate on validation set
    """
    # Create confidence matrix
    confidence_matrix = (item_user_matrix * alpha).astype('float64')
    
    # Train model
    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42
    )
    model.fit(confidence_matrix)
    
    # Generate predictions
    predicted = []
    actual = []
    
    for user in eval_users:
        if user not in user_mapping:
            predicted.append([])
            actual.append(ground_truth[user])
            continue
        
        user_idx = user_mapping[user]
        ids, scores = model.recommend(
            user_idx,
            confidence_matrix[user_idx],
            N=10,
            filter_already_liked_items=True
        )
        
        recommended_items = [item_list[idx] for idx in ids]
        predicted.append(recommended_items)
        actual.append(ground_truth[user])
    
    # Calculate MAP@10
    score = mapk(actual, predicted, k=10)
    return score

print("Evaluation function for hyperparameter tuning defined.")

Evaluation function for hyperparameter tuning defined.


In [15]:
print("\n3. Starting hyperparameter search...")
print("   This may take several minutes...\n")

best_score = 0
best_params = {}
results = []

total_combinations = (len(param_grid['factors']) * len(param_grid['regularization']) * 
                     len(param_grid['iterations']) * len(param_grid['alpha']))
current_combination = 0

start_time = time.time()

for factors in param_grid['factors']:
    for regularization in param_grid['regularization']:
        for iterations in param_grid['iterations']:
            for alpha in param_grid['alpha']:
                current_combination += 1
                
                print(f"   [{current_combination}/{total_combinations}] Testing: factors={factors}, "
                      f"reg={regularization}, iter={iterations}, alpha={alpha}")
                
                score = evaluate_als_params(
                    factors, regularization, iterations, alpha,
                    item_user_sparse, user_id_to_idx, all_items_als,
                    tuning_users, validation_truth
                )
                
                results.append({
                    'factors': factors,
                    'regularization': regularization,
                    'iterations': iterations,
                    'alpha': alpha,
                    'map@10': score
                })
                
                print(f"      MAP@10: {score:.6f}")
                
                if score > best_score:
                    best_score = score
                    best_params = {
                        'factors': factors,
                        'regularization': regularization,
                        'iterations': iterations,
                        'alpha': alpha
                    }
                    print(f"      *** New best score! ***")

print(f"\n   Total tuning time: {time.time() - start_time:.2f} seconds")
print(f"\n4. Best Parameters Found:")
for param, value in best_params.items():
    print(f"   {param}: {value}")
print(f"   Best MAP@10 (on tuning set): {best_score:.6f}")


3. Starting hyperparameter search...
   This may take several minutes...

   [1/81] Testing: factors=50, reg=0.01, iter=15, alpha=1.0


100%|██████████| 15/15 [01:00<00:00,  4.02s/it]


      MAP@10: 0.000015
      *** New best score! ***
   [2/81] Testing: factors=50, reg=0.01, iter=15, alpha=10.0


100%|██████████| 15/15 [01:02<00:00,  4.20s/it]


      MAP@10: 0.000109
      *** New best score! ***
   [3/81] Testing: factors=50, reg=0.01, iter=15, alpha=40.0


...

      MAP@10: 0.000044
   [6/81] Testing: factors=50, reg=0.01, iter=20, alpha=40.0


100%|██████████| 20/20 [01:14<00:00,  3.73s/it]


      MAP@10: 0.000109
   [7/81] Testing: factors=50, reg=0.01, iter=25, alpha=1.0


 52%|█████▏    | 13/25 [00:50<00:47,  3.92s/it]


KeyboardInterrupt: 

In [ ]:
print("\n5. Training final ALS model with best parameters on full validation set...")

# Create confidence matrix with best alpha
als_confidence_tuned = (item_user_sparse * best_params['alpha']).astype('float64')

# Train model with best parameters
model_als_tuned = AlternatingLeastSquares(
    factors=best_params['factors'],
    regularization=best_params['regularization'],
    iterations=best_params['iterations'],
    random_state=42
)

start_time = time.time()
model_als_tuned.fit(als_confidence_tuned)
print(f"   Training time: {time.time() - start_time:.2f} seconds")

In [ ]:
print("\n6. Generating recommendations for all validation users...")

predicted_list_als_tuned = []
actual_list_als_tuned = []

start_time = time.time()
for i, user in enumerate(validation_users):
    if (i + 1) % 1000 == 0:
        print(f"   Processed {i + 1}/{len(validation_users)} users...")
    
    if user not in user_id_to_idx:
        predicted_list_als_tuned.append([])
        actual_list_als_tuned.append(validation_truth[user])
        continue
    
    user_idx = user_id_to_idx[user]
    
    ids, scores = model_als_tuned.recommend(
        user_idx,
        als_confidence_tuned[user_idx],
        N=10,
        filter_already_liked_items=True
    )
    
    recommended_items = [all_items_als[idx] for idx in ids]
    
    predicted_list_als_tuned.append(recommended_items)
    actual_list_als_tuned.append(validation_truth[user])

print(f"   Time taken: {time.time() - start_time:.2f} seconds")

# Evaluate tuned ALS
print("\n7. Evaluating tuned ALS model...")
mapk_score_als_tuned = mapk(actual_list_als_tuned, predicted_list_als_tuned, k=10)

print(f"\n{'='*60}")
print(f"ALS (Tuned) Results:")
print(f"MAP@10 Score: {mapk_score_als_tuned:.6f}")
print(f"{'='*60}\n")

## Comparison 

In [ ]:
print("\n" + "=" * 70)
print("FINAL COMPARISON: ALS MODELS")
print("=" * 70)

comparison_df = pd.DataFrame({
    'Model': ['ALS (Default)', 'ALS (Tuned)'],
    'MAP@10': [mapk_score_als_default, mapk_score_als_tuned]
})

# Calculate improvement from default to tuned
comparison_df['Improvement over Default (%)'] = (
    (comparison_df['MAP@10'] - mapk_score_als_default) / mapk_score_als_default * 100
)

print("\n", comparison_df.to_string(index=False))
print("\n" + "=" * 70)

# Determine best model
best_model = comparison_df.loc[comparison_df['MAP@10'].idxmax()]
print(f"\n🏆 Best Model: {best_model['Model']}")
print(f"   MAP@10 Score: {best_model['MAP@10']:.6f}")
if best_model['Model'] == 'ALS (Tuned)':

    print(f"   Improvement over Default: {best_model['Improvement over Default (%)']:.2f}%")print("\n" + "=" * 70)

## Simple Visualization - Model Comparison

In [ ]:
# Simple and clean visualization comparing the 2 ALS models
plt.figure(figsize=(10, 6))

# Define colors for each model
colors = ['#4ECDC4', '#45B7D1']  # Teal, Blue

# Create bar chart
bars = plt.bar(comparison_df['Model'], comparison_df['MAP@10'], 
               color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add title and labels
plt.title('ALS Model Performance Comparison - MAP@10 Score', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('MAP@10 Score', fontsize=13, fontweight='bold')
plt.xlabel('Model', fontsize=13, fontweight='bold')

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.4f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add grid for easier reading
plt.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.7)

# Set y-axis to start from 0
plt.ylim(bottom=0, top=max(comparison_df['MAP@10']) * 1.15)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")

In [ ]:
# Additional visualization: Line plot showing progression from Default to Tuned
plt.figure(figsize=(10, 6))

# Create line plot
plt.plot(comparison_df['Model'], comparison_df['MAP@10'], 
         marker='o', markersize=12, linewidth=3, color='#2ECC71', 
         markerfacecolor='#E74C3C', markeredgewidth=2, markeredgecolor='black')

# Add title and labels
plt.title('ALS Performance: Default vs Tuned', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('MAP@10 Score', fontsize=13, fontweight='bold')
plt.xlabel('Model', fontsize=13, fontweight='bold')

# Add value labels next to each point
for i, (model, score) in enumerate(zip(comparison_df['Model'], comparison_df['MAP@10'])):
    plt.text(i, score + max(comparison_df['MAP@10']) * 0.02, 
             f'{score:.4f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# Add grid
plt.grid(True, alpha=0.3, linestyle='--', linewidth=0.7)

# Set y-axis limits
plt.ylim(bottom=0, top=max(comparison_df['MAP@10']) * 1.15)

plt.tight_layout()
plt.show()

print("✓ Line plot visualization complete!")

## Visualization of Results

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot of MAP@10 scores
ax1 = axes[0]
colors = ['#4ECDC4', '#45B7D1']
bars = ax1.bar(comparison_df['Model'], comparison_df['MAP@10'], color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('MAP@10 Score', fontsize=12, fontweight='bold')
ax1.set_title('ALS Model Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim([0, max(comparison_df['MAP@10']) * 1.2])
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.6f}',
             ha='center', va='bottom', fontweight='bold', fontsize=10)

# Improvement from Default to Tuned
ax2 = axes[1]
colors_improvement = ['gray', '#2ecc71']
bars2 = ax2.bar(comparison_df['Model'], comparison_df['Improvement over Default (%)'], 
                color=colors_improvement, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Improvement over Default (%)', fontsize=12, fontweight='bold')
ax2.set_title('Tuning Impact', fontsize=14, fontweight='bold')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}%',
             ha='center', va='bottom' if height >= 0 else 'top', 
             fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print("Visualization complete!")

## Generate Recommendations for Test Users

Now we'll use the best model to generate recommendations for the target test users.

In [ ]:
print("\nGenerating recommendations for test users using the best model...")

# Retrain the best model on the FULL training data (including validation)
print("\n1. Retraining best model on full training data...")

# Recreate mappings with full data
all_users_full = sorted(train['user_id'].unique())
all_items_full = sorted(train['item_id'].unique())

user_id_to_idx_full = {user_id: idx for idx, user_id in enumerate(all_users_full)}
item_id_to_idx_full = {item_id: idx for idx, item_id in enumerate(all_items_full)}

# Create sparse matrix with full data
rows_full = train['item_id'].map(item_id_to_idx_full).values
cols_full = train['user_id'].map(user_id_to_idx_full).values
data_full = np.ones(len(train))

item_user_sparse_full = csr_matrix((data_full, (rows_full, cols_full)),
                                   shape=(len(all_items_full), len(all_users_full)))

# Create confidence matrix with best alpha
als_confidence_full = (item_user_sparse_full * best_params['alpha']).astype('float64')

# Train final model
model_final = AlternatingLeastSquares(
    factors=best_params['factors'],
    regularization=best_params['regularization'],
    iterations=best_params['iterations'],
    random_state=42
)

start_time = time.time()
model_final.fit(als_confidence_full)
print(f"   Training time: {time.time() - start_time:.2f} seconds")

print(f"\n2. Trained on {len(all_users_full)} users and {len(all_items_full)} items")

In [ ]:
print("\n3. Generating recommendations for target test users...")

test_users = data_target['user_id'].tolist()
print(f"   Number of test users: {len(test_users)}")

# Generate recommendations
submission_data = []

for user in test_users:
    if user not in user_id_to_idx_full:
        # Cold start user - recommend most popular items
        popular_items = train['item_id'].value_counts().head(10).index.tolist()
        for rank, item in enumerate(popular_items, 1):
            submission_data.append({
                'user_id': user,
                'item_id': item,
                'rank': rank
            })
    else:
        user_idx = user_id_to_idx_full[user]
        
        # Get recommendations
        ids, scores = model_final.recommend(
            user_idx,
            als_confidence_full[user_idx],
            N=10,
            filter_already_liked_items=True
        )
        
        # Convert to item IDs
        recommended_items = [all_items_full[idx] for idx in ids]
        
        for rank, item in enumerate(recommended_items, 1):
            submission_data.append({
                'user_id': user,
                'item_id': item,
                'rank': rank
            })

# Create submission dataframe
submission_df = pd.DataFrame(submission_data)
print(f"\n4. Generated {len(submission_df)} recommendations")
print(f"   Shape: {submission_df.shape}")
print("\nFirst few recommendations:")
print(submission_df.head(15))

In [ ]:
# Save to CSV
output_filename = 'als_tuned_submission.csv'
submission_df.to_csv(output_filename, index=False)
print(f"\n✓ Submission file saved as: {output_filename}")

# Also save comparison results
comparison_df.to_csv('model_comparison_results.csv', index=False)
print(f"✓ Model comparison saved as: model_comparison_results.csv")

# Save hyperparameter tuning results
tuning_results_df = pd.DataFrame(results)
tuning_results_df = tuning_results_df.sort_values('map@10', ascending=False)
tuning_results_df.to_csv('hyperparameter_tuning_results.csv', index=False)
print(f"✓ Hyperparameter tuning results saved as: hyperparameter_tuning_results.csv")

print("\n" + "="*70)
print("ALL MODELS COMPLETED SUCCESSFULLY!")
print("="*70)

## Summary

### Models Implemented:

1. **ALS (Alternating Least Squares) - Default Parameters**
   - Matrix factorization for implicit feedback
   - Default parameters: factors=50, regularization=0.01, iterations=15, alpha=1.0
   - Baseline performance for comparison

2. **ALS - Tuned Parameters**
   - Hyperparameter optimization using grid search
   - Tests combinations of factors, regularization, iterations, and alpha
   - Selects best parameters based on MAP@10 score
   - Improved performance through systematic parameter tuning

### Evaluation Metric:
- **MAP@10** (Mean Average Precision at 10)
- Measures ranking quality of top-10 recommendations
- Higher scores indicate better recommendation quality

### Output Files:
- `als_tuned_submission.csv` - Final recommendations for test users
- `model_comparison_results.csv` - Performance comparison of both ALS models
- `hyperparameter_tuning_results.csv` - Detailed tuning results